In [ ]:
# ================================================================
# SEL 0 -- Instalasi Dependensi
# ================================================================
# IndoBERT & training stack (identik V1)
!pip install -q kagglehub transformers datasets accelerate sentencepiece scikit-learn

# BERTopic stack (untuk Metode 3)
!pip install -q umap-learn hdbscan bertopic sentence-transformers

# Hugging Face Hub
!pip install -q huggingface_hub

print("Instalasi selesai.")


In [ ]:
# ================================================================
# SEL 1 -- Import Library
# ================================================================
import os, gc, random, re, pickle, warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback, set_seed,
)
from datasets import Dataset
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    f1_score, ConfusionMatrixDisplay,
)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())


## Konfigurasi

> **CONSTRAINT**: `train_batch_size=96`, `eval_batch_size=384`, `max_length=256`
> tidak boleh diubah. Topic modelling adalah **post-training**.


In [ ]:
# ================================================================
# SEL 2 -- Config Training IndoBERT (identik V1 — jangan ubah)
# ================================================================
@dataclass
class Config:
    path_cnn    : str   = "/content/dataset/Summarized_CNN.csv"
    path_detik  : str   = "/content/dataset/Summarized_Detik.csv"
    path_kompas : str   = "/content/dataset/Summarized_Kompas.csv"
    path_tbh    : str   = "/content/dataset/Summarized_TurnBackHoax.csv"
    path_extra  : str   = "/content/dataset/Summarized_2020+.csv"
    model_name       : str   = "indolem/indobert-base-uncased"
    max_length       : int   = 256   # CONSTRAINT
    train_batch_size : int   = 96    # CONSTRAINT
    eval_batch_size  : int   = 384   # CONSTRAINT
    grad_accumulation: int   = 2
    learning_rate    : float = 2e-5
    weight_decay     : float = 0.01
    num_epochs       : int   = 3
    seed             : int   = 42
    balance_minority : bool  = True
    output_dir       : str   = "indobert_hoax_model_v3"

cfg = Config()
set_seed(cfg.seed)
torch.backends.cudnn.deterministic = True
print(f"Config: model={cfg.model_name} | max_len={cfg.max_length} | batch={cfg.train_batch_size}")


In [ ]:
# ================================================================
# SEL 3 -- Config Topic Modelling V2 (Post-Training)
# ================================================================
# Flag aktivasi — dapat di-set False tanpa mengubah training
AKTIFKAN_TFIDF    = True   # Metode 1: selalu aktif, tanpa artefak
AKTIFKAN_LDA      = True   # Metode 2: sklearn, ringan
AKTIFKAN_BERTOPIC = True   # Metode 3: embedding semantik

# LDA
LDA_N_TOPICS         = 10
LDA_MAX_ITER         = 20
LDA_MAX_FEATURES     = 3000
LDA_MIN_DF           = 3
LDA_RANDOM_STATE     = cfg.seed
DIR_OUTPUT_LDA       = "/content/lda_model_v1"
REPO_LDA_HF          = ""   # <- isi 'username/repo-name'

# BERTopic
MODEL_EMBEDDING_BT = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
UKURAN_BATCH_EMBED = 32
MAKS_DOKUMEN_BT    = None
BERTOPIC_PROB      = False
NR_TOPIK           = "auto"
DIR_OUTPUT_BT      = "/content/bertopic_model_v1"
REPO_BT_HF         = ""     # <- isi 'username/repo-name'
UPLOAD_BT          = False
SEED_BT            = cfg.seed

TOPIC_KEYWORDS_TOPK = 5

print(f"TF-IDF={AKTIFKAN_TFIDF} | LDA={AKTIFKAN_LDA} | BERTopic={AKTIFKAN_BERTOPIC}")


In [ ]:
# ================================================================
# SEL 4 -- Unduh Dataset dari Kaggle
# ================================================================
import kagglehub, shutil
kaggle_cache = kagglehub.dataset_download("fjrmhri/dataset-berita")
LOCAL_DIR = Path("/content/dataset")
LOCAL_DIR.mkdir(exist_ok=True)
for f in Path(kaggle_cache).rglob("*.csv"):
    dest = LOCAL_DIR / f.name
    if not dest.exists():
        shutil.copy2(f, dest)
print("Dataset tersedia di:", LOCAL_DIR)


In [ ]:
# ================================================================
# SEL 5 -- Pemuatan Data & Pra-pemrosesan
# ================================================================
KOLOM = ["url","judul","tanggal","isi_berita","Narasi","Clean Narasi","hoax","summary"]

def muat_semua(c: Config) -> pd.DataFrame:
    frames = []
    for path, src in [(c.path_cnn,"CNN"),(c.path_detik,"Detik"),
                      (c.path_kompas,"Kompas"),(c.path_tbh,"TBH"),(c.path_extra,"Extra")]:
        p = Path(path)
        if not p.exists(): continue
        df = pd.read_csv(p, low_memory=False)
        df = df[[c for c in KOLOM if c in df.columns]].copy()
        df["sumber"] = src; frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def bersihkan(df_raw: pd.DataFrame) -> pd.DataFrame:
    df = df_raw.copy()
    def pilih_teks(row):
        for col in ["Clean Narasi","Narasi","isi_berita","summary","judul"]:
            if col in row and pd.notna(row[col]) and str(row[col]).strip():
                return str(row[col]).strip()
        return None
    df["text"] = df.apply(pilih_teks, axis=1)
    df = df.dropna(subset=["text"])
    df = df[df["text"].str.len() >= 20]
    def norm_label(v):
        if pd.isna(v): return None
        s = str(v).strip().lower()
        if s in {"1","true","hoax","hoaks"}: return 1
        if s in {"0","false","not_hoax","bukan"}: return 0
        try: return int(float(s))
        except: return None
    df["label"] = df["hoax"].apply(norm_label)
    df = df.dropna(subset=["label"])
    df["label"] = df["label"].astype(int)
    return df[df["label"].isin([0,1])].drop_duplicates(subset=["text"]).reset_index(drop=True)

df_raw   = muat_semua(cfg)
df_bersih = bersihkan(df_raw)
print(f"Bersih: {len(df_bersih):,} | label: {df_bersih['label'].value_counts().to_dict()}")


In [ ]:
# ================================================================
# SEL 6 -- Split Data & Balancing
# ================================================================
train_df, temp_df = train_test_split(df_bersih, test_size=0.30,
                                     stratify=df_bersih["label"], random_state=cfg.seed)
val_df, test_df   = train_test_split(temp_df, test_size=0.50,
                                     stratify=temp_df["label"], random_state=cfg.seed)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

# Simpan pra-oversampling untuk topic modelling (bebas duplikat)
train_df_pra_oversampling = train_df.copy()

if cfg.balance_minority:
    n_maj = (train_df["label"] == 0).sum()
    min_df = train_df[train_df["label"] == 1]
    if len(min_df) < n_maj:
        extra = min_df.sample(n=n_maj - len(min_df), replace=True, random_state=cfg.seed)
        train_df = pd.concat([train_df, extra]).sample(frac=1, random_state=cfg.seed).reset_index(drop=True)

print(f"Train: {len(train_df):,} (pra: {len(train_df_pra_oversampling):,}) | Val: {len(val_df):,} | Test: {len(test_df):,}")


In [ ]:
# ================================================================
# SEL 7 -- Tokenisasi & Training IndoBERT
# ================================================================
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=cfg.max_length)

def df_to_hf(df):
    ds = Dataset.from_dict({"text": df["text"].tolist(), "label": df["label"].tolist()})
    return ds.map(tokenize_batch, batched=True, batch_size=512, remove_columns=["text"])

ds_train = df_to_hf(train_df)
ds_val   = df_to_hf(val_df)
ds_test  = df_to_hf(test_df)

label2id = {"not_hoax": 0, "hoax": 1}
id2label = {v: k for k, v in label2id.items()}

model_clf = AutoModelForSequenceClassification.from_pretrained(
    cfg.model_name, num_labels=2, label2id=label2id, id2label=id2label,
    ignore_mismatched_sizes=True,
)

def hitung_metrik(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    return {"accuracy": accuracy_score(labels, preds), "f1": f, "precision": p, "recall": r}

training_args = TrainingArguments(
    output_dir=cfg.output_dir, num_train_epochs=cfg.num_epochs,
    per_device_train_batch_size=cfg.train_batch_size,
    per_device_eval_batch_size=cfg.eval_batch_size,
    gradient_accumulation_steps=cfg.grad_accumulation,
    learning_rate=cfg.learning_rate, weight_decay=cfg.weight_decay,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="f1",
    fp16=torch.cuda.is_available(), seed=cfg.seed,
    logging_steps=100, report_to="none",
)

trainer = Trainer(
    model=model_clf, args=training_args,
    train_dataset=ds_train, eval_dataset=ds_val,
    tokenizer=tokenizer, compute_metrics=hitung_metrik,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Mulai training IndoBERT...")
trainer.train()
print("Training selesai. Evaluasi test:")
print(trainer.evaluate(ds_test))


In [ ]:
# ================================================================
# SEL 8 -- Laporan & Confusion Matrix IndoBERT
# ================================================================
def laporan_detail(trainer, dataset, split_name):
    out = trainer.predict(dataset)
    y_true = out.label_ids
    y_pred = np.argmax(out.predictions, axis=-1)
    print(f"\n{'='*50}\n  {split_name}\n{'='*50}")
    print(classification_report(y_true, y_pred, target_names=["not_hoax","hoax"]))
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(4,3))
    ConfusionMatrixDisplay(cm, display_labels=["not_hoax","hoax"]).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"CM — {split_name}")
    plt.tight_layout()
    plt.savefig(f"/content/confusion_matrix_{split_name.lower()}.png", dpi=120)
    plt.show()
    return y_true, y_pred

laporan_detail(trainer, ds_val,  "Validation")
laporan_detail(trainer, ds_test, "Test")


In [ ]:
# ================================================================
# SEL 9 -- Simpan IndoBERT & Upload ke HF (repo TERPISAH)
# ================================================================
os.makedirs(cfg.output_dir, exist_ok=True)
trainer.model.save_pretrained(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)
print(f"Model disimpan: {cfg.output_dir}")

# Upload IndoBERT — gunakan repo BERBEDA dari topic model artifacts
REPO_INDOBERT_HF = ""   # <- ganti dengan 'username/repo-indohoax'
if REPO_INDOBERT_HF:
    from huggingface_hub import login, HfApi
    try:
        from google.colab import userdata
        login(token=userdata.get("HF_TOKEN"))
    except: login()
    api = HfApi()
    api.create_repo(repo_id=REPO_INDOBERT_HF, private=False, exist_ok=True)
    api.upload_folder(folder_path=cfg.output_dir, repo_id=REPO_INDOBERT_HF,
                      repo_type="model", commit_message="IndoBERT Hoax V2")
    print(f"IndoBERT diupload: https://huggingface.co/{REPO_INDOBERT_HF}")
else:
    print("REPO_INDOBERT_HF kosong — upload dilewati.")


---
## Topic Modelling V2 (Post-Training)

Semua sel berikut berjalan **setelah** training selesai.
Tidak satu pun yang menyentuh gradient atau loop training.

| # | Metode | Library | Artefak | Kecepatan |
|---|--------|---------|---------|-----------|
| 1 | **TF-IDF** | sklearn | ❌ tidak perlu | ⚡ sangat cepat |
| 2 | **LDA** | sklearn | `lda_model.pkl` | ⚡ cepat |
| 3 | **BERTopic** | bertopic | `bertopic_model_v1/` | 🐢 lambat (embedding) |

**Rekomendasi metode ke-3: LDA** — karena bagian dari sklearn (sudah terpasang),
deterministik, artefak ringan, menghasilkan distribusi topik (berguna untuk evaluasi proxy),
dan mudah deploy sebagai pickle tunggal.


In [ ]:
# ================================================================
# SEL 10 -- Setup Inferensi Pasca-Training
# ================================================================
import re as _re
WS_RE    = _re.compile(r"\s+")
TOKEN_RE = _re.compile(r"[a-zA-Z]{3,}")
PARA_RE  = _re.compile(r"(?:\r?\n){2,}")
SENT_RE  = _re.compile(r'[^.!?]+(?:[.!?]+(?:["\)\]]+)?)|[^.!?]+$')

ID_STOPWORDS = {
    "yang","dan","atau","di","ke","dari","untuk","dengan","pada","adalah",
    "itu","ini","dalam","sebagai","karena","juga","agar","oleh","saat","akan",
    "telah","sudah","tidak","iya","ya","kita","mereka","kami","anda","hingga",
    "lebih","masih","dapat","bisa","setelah","sebelum","tersebut","terhadap",
    "disebut","menurut","para","sebuah","adanya","yakni","bahwa","the",
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
try:
    _infer_tok = tokenizer
    _infer_mdl = model_clf.to(DEVICE); _infer_mdl.eval()
    print("Menggunakan model dari training session.")
except NameError:
    _infer_tok = AutoTokenizer.from_pretrained(cfg.output_dir)
    _infer_mdl = AutoModelForSequenceClassification.from_pretrained(cfg.output_dir).to(DEVICE)
    _infer_mdl.eval()
    print("Model dimuat ulang dari disk.")

def infer_proba(texts, batch_size=64):
    prepared = [WS_RE.sub(" ", str(t)).strip() or "[EMPTY]" for t in texts]
    results = []
    for i in range(0, len(prepared), batch_size):
        chunk = prepared[i:i+batch_size]
        enc = _infer_tok(chunk, padding=True, truncation=True,
                         max_length=cfg.max_length, return_tensors="pt")
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.inference_mode():
            probs = torch.softmax(_infer_mdl(**enc).logits, dim=-1).cpu().numpy()
        for row in probs:
            results.append({"not_hoax": float(row[0]), "hoax": float(row[1])})
    return results

def pisahkan_par(teks):
    raw = str(teks).strip()
    if not raw: return []
    parts = [p.strip() for p in PARA_RE.split(raw) if p.strip()]
    if len(parts) <= 1 and "\n" in raw:
        lines = [p.strip() for p in raw.splitlines() if p.strip()]
        if len(lines) > 1: return lines
    return parts or [raw]

print("Utilitas inferensi siap.")


### Metode 1 — TF-IDF

In [ ]:
# ================================================================
# SEL 11 -- Metode 1: TF-IDF Keyword Extraction (identik V1)
# ================================================================
def ekstrak_topik_tfidf(paragraph_texts, topk=TOPIC_KEYWORDS_TOPK, max_features=2000):
    if not paragraph_texts: return []
    hasil = []
    try:
        vec = TfidfVectorizer(lowercase=True, ngram_range=(1,2), max_features=max_features,
                              token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",
                              stop_words=list(ID_STOPWORDS))
        matrix = vec.fit_transform(paragraph_texts)
        features = vec.get_feature_names_out()
        for i in range(matrix.shape[0]):
            row = matrix.getrow(i)
            if row.nnz == 0:
                tokens = [t for t in TOKEN_RE.findall(paragraph_texts[i].lower()) if t not in ID_STOPWORDS]
                kws = list(dict.fromkeys(tokens))[:topk] or ["topik_umum"]
                score = 0.0
            else:
                pairs = sorted(zip(row.indices, row.data), key=lambda x: -x[1])[:topk]
                kws   = [features[idx] for idx,_ in pairs] or ["topik_umum"]
                score = float(np.mean([v for _,v in pairs]))
            hasil.append({"label": " / ".join(kws[:2]), "score": round(score,6), "keywords": kws})
    except Exception as e:
        for text in paragraph_texts:
            tokens = [t for t in TOKEN_RE.findall(text.lower()) if t not in ID_STOPWORDS]
            kws = list(dict.fromkeys(tokens))[:topk] or ["topik_umum"]
            hasil.append({"label": " / ".join(kws[:2]), "score": 0.0, "keywords": kws})
    return hasil

if AKTIFKAN_TFIDF:
    demo = ekstrak_topik_tfidf(["Bantuan pemerintah untuk warga.","Kementerian membantah hoaks."])
    print("[TF-IDF] Demo:", [d["label"] for d in demo])


### Metode 2 — LDA

**Alasan memilih LDA:**
- Bagian dari `sklearn` → tidak ada dependensi baru, mudah deploy
- Deterministik dengan `random_state` → reproducible
- Output berupa distribusi probabilitas topik → berguna untuk evaluasi proxy LogReg
- Artefak ringan: satu file `.pkl` berisi `{vectorizer, lda}`
- Inference cepat: hanya `vectorizer.transform` + `lda.transform`

**Trade-off:**
- Tidak menangkap semantik kata (berbeda dari BERTopic)
- Jumlah topik harus ditentukan manual (LDA_N_TOPICS)


In [ ]:
# ================================================================
# SEL 12 -- Metode 2: Fit LDA
# ================================================================
if not AKTIFKAN_LDA:
    model_lda = None; vec_lda = None
    print("AKTIFKAN_LDA=False.")
else:
    print(f"Fit LDA pada {len(train_df_pra_oversampling):,} dokumen (pra-oversampling)...")
    vec_lda = CountVectorizer(
        max_features=LDA_MAX_FEATURES, min_df=LDA_MIN_DF,
        stop_words=list(ID_STOPWORDS),
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",
    )
    corpus_lda = train_df_pra_oversampling["text"].astype(str).tolist()
    dtm_lda    = vec_lda.fit_transform(corpus_lda)
    print(f"DTM: {dtm_lda.shape}")

    model_lda = LatentDirichletAllocation(
        n_components=LDA_N_TOPICS, max_iter=LDA_MAX_ITER,
        random_state=LDA_RANDOM_STATE, n_jobs=-1, verbose=0,
    )
    model_lda.fit(dtm_lda)
    print(f"LDA fit. n_topics={LDA_N_TOPICS}")

    features_lda = vec_lda.get_feature_names_out()
    print("\nTop-10 kata per topik LDA:")
    for t_idx, topic in enumerate(model_lda.components_):
        words = [features_lda[i] for i in topic.argsort()[-10:][::-1]]
        print(f"  Topik {t_idx:2d}: {' | '.join(words)}")


In [ ]:
# ================================================================
# SEL 13 -- Metode 2: LDA Inferensi
# ================================================================
def ekstrak_topik_lda(paragraph_texts, topk=TOPIC_KEYWORDS_TOPK):
    if model_lda is None or vec_lda is None:
        return ekstrak_topik_tfidf(paragraph_texts, topk=topk)
    if not paragraph_texts: return []
    try:
        dtm   = vec_lda.transform(paragraph_texts)
        dists = model_lda.transform(dtm)
        feats = vec_lda.get_feature_names_out()
        hasil = []
        for i, dist in enumerate(dists):
            dom  = int(np.argmax(dist))
            score = float(dist[dom])
            widx = model_lda.components_[dom].argsort()[-topk:][::-1]
            kws  = [str(feats[j]) for j in widx] or ["topik_umum"]
            hasil.append({"label": " / ".join(kws[:2]), "score": round(score,6),
                          "keywords": kws, "topic_id": dom, "dist": dist.tolist()})
        return hasil
    except Exception as e:
        print(f"[WARN] LDA err: {e}")
        return ekstrak_topik_tfidf(paragraph_texts, topk=topk)

if AKTIFKAN_LDA and model_lda is not None:
    demo_lda = ekstrak_topik_lda(["Bantuan pemerintah untuk warga.","Kementerian membantah hoaks."])
    print("[LDA] Demo:")
    for d in demo_lda: print(f"  topik #{d['topic_id']} | {d['label']} | score={d['score']:.3f}")


In [ ]:
# ================================================================
# SEL 14 -- Simpan & Upload Artefak LDA (repo TERPISAH dari IndoBERT)
# ================================================================
if AKTIFKAN_LDA and model_lda is not None:
    os.makedirs(DIR_OUTPUT_LDA, exist_ok=True)
    artefak_lda = {"vectorizer": vec_lda, "lda": model_lda,
                   "n_topics": LDA_N_TOPICS, "versi": "V2"}
    pkl_path = os.path.join(DIR_OUTPUT_LDA, "lda_model.pkl")
    with open(pkl_path, "wb") as f:
        pickle.dump(artefak_lda, f)
    print(f"LDA disimpan: {pkl_path}")
    if REPO_LDA_HF:
        from huggingface_hub import HfApi
        try:
            from google.colab import userdata; from huggingface_hub import login
            login(token=userdata.get("HF_TOKEN"))
        except: pass
        api = HfApi()
        api.create_repo(repo_id=REPO_LDA_HF, private=False, exist_ok=True)
        api.upload_folder(folder_path=DIR_OUTPUT_LDA, repo_id=REPO_LDA_HF,
                          repo_type="model", commit_message="LDA V2 topic model")
        print(f"LDA diupload: https://huggingface.co/{REPO_LDA_HF}")
    else:
        print("REPO_LDA_HF kosong — upload LDA dilewati.")


### Metode 3 — BERTopic

In [ ]:
# ================================================================
# SEL 15 -- Fit BERTopic (Post-Training)
# ================================================================
if not AKTIFKAN_BERTOPIC:
    model_bt = None; print("AKTIFKAN_BERTOPIC=False.")
else:
    from bertopic import BERTopic
    from sentence_transformers import SentenceTransformer
    from umap import UMAP; from hdbscan import HDBSCAN

    corpus_bt = train_df_pra_oversampling["text"].astype(str).tolist()
    if MAKS_DOKUMEN_BT:
        corpus_bt = random.Random(SEED_BT).sample(corpus_bt, min(MAKS_DOKUMEN_BT, len(corpus_bt)))
    print(f"BERTopic: {len(corpus_bt):,} dokumen")

    dev = "cuda" if torch.cuda.is_available() else "cpu"
    emb_mdl = SentenceTransformer(MODEL_EMBEDDING_BT, device=dev)
    emb_bt  = emb_mdl.encode(corpus_bt, batch_size=UKURAN_BATCH_EMBED,
                              show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
    del emb_mdl; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    umap_m = UMAP(n_neighbors=15, n_components=5, min_dist=0.0,
                  metric="cosine", random_state=SEED_BT, low_memory=True)
    hdb_m  = HDBSCAN(min_cluster_size=15, metric="euclidean",
                     cluster_selection_method="eom", prediction_data=True)
    vec_m  = CountVectorizer(stop_words=list(ID_STOPWORDS),
                             token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b")

    model_bt = BERTopic(umap_model=umap_m, hdbscan_model=hdb_m, vectorizer_model=vec_m,
                        nr_topics=NR_TOPIK, calculate_probabilities=BERTOPIC_PROB, verbose=True)
    bt_topics, _ = model_bt.fit_transform(corpus_bt, embeddings=emb_bt)
    info_bt = model_bt.get_topic_info()
    print(f"BERTopic selesai. Topik (incl. outlier): {len(info_bt)}")
    print(info_bt.head(8).to_string())


In [ ]:
# ================================================================
# SEL 16 -- BERTopic Inferensi
# ================================================================
_emb_infer = None

def ekstrak_topik_bertopic(paragraph_texts, topk=TOPIC_KEYWORDS_TOPK):
    global _emb_infer
    if model_bt is None: return ekstrak_topik_tfidf(paragraph_texts, topk=topk)
    if not paragraph_texts: return []
    try:
        from sentence_transformers import SentenceTransformer
        if _emb_infer is None:
            dev = "cuda" if torch.cuda.is_available() else "cpu"
            _emb_infer = SentenceTransformer(MODEL_EMBEDDING_BT, device=dev)
        emb  = _emb_infer.encode(paragraph_texts, batch_size=16,
                                 show_progress_bar=False, convert_to_numpy=True,
                                 normalize_embeddings=True)
        tids, _ = model_bt.transform(paragraph_texts, embeddings=emb)
        hasil = []
        for t_id in tids:
            t_id = int(t_id)
            words = model_bt.get_topic(t_id) or []
            kws   = [w for w,_ in words[:topk]] or ["topik_umum"]
            score = float(words[0][1]) if words else 0.0
            hasil.append({"label": " / ".join(kws[:2]), "score": round(max(0.0,score),6),
                          "keywords": kws, "topic_id": t_id})
        return hasil
    except Exception as e:
        print(f"[WARN] BERTopic err: {e}")
        return ekstrak_topik_tfidf(paragraph_texts, topk=topk)

if AKTIFKAN_BERTOPIC and model_bt is not None:
    demo_bt = ekstrak_topik_bertopic(["Bantuan pemerintah untuk warga.","Kementerian membantah hoaks."])
    print("[BERTopic] Demo:", [(d["topic_id"], d["label"]) for d in demo_bt])


In [ ]:
# ================================================================
# SEL 17 -- Simpan & Upload BERTopic
# ================================================================
if AKTIFKAN_BERTOPIC and model_bt is not None:
    os.makedirs(DIR_OUTPUT_BT, exist_ok=True)
    model_bt.save(DIR_OUTPUT_BT, serialization="pickle", save_ctfidf=True)
    print(f"BERTopic disimpan: {DIR_OUTPUT_BT}")
    if UPLOAD_BT and REPO_BT_HF:
        from huggingface_hub import HfApi
        try:
            from google.colab import userdata; from huggingface_hub import login
            login(token=userdata.get("HF_TOKEN"))
        except: pass
        api = HfApi()
        api.create_repo(repo_id=REPO_BT_HF, private=False, exist_ok=True)
        api.upload_folder(folder_path=DIR_OUTPUT_BT, repo_id=REPO_BT_HF,
                          repo_type="model", commit_message="BERTopic V2")
        print(f"BERTopic diupload: https://huggingface.co/{REPO_BT_HF}")
    else:
        print("Upload BERTopic dilewati.")


---
## Demo Inferensi 3 Metode (Side-by-Side)


In [ ]:
# ================================================================
# SEL 18 -- Router Topik & Demo
# ================================================================
def ekstrak_topik(paragraph_texts, metode="tfidf", topk=TOPIC_KEYWORDS_TOPK):
    """Router utama — pilih metode ekstraksi topik."""
    if metode == "lda":      return ekstrak_topik_lda(paragraph_texts, topk=topk)
    if metode == "bertopic": return ekstrak_topik_bertopic(paragraph_texts, topk=topk)
    return ekstrak_topik_tfidf(paragraph_texts, topk=topk)

CONTOH_TEKS = """
Beredar unggahan media sosial yang menyebut pemerintah membagikan bantuan tunai \
tanpa syarat lewat tautan tertentu. Banyak akun meminta warga mengisi data pribadi.

Kementerian Sosial menegaskan bahwa informasi tersebut tidak benar. \
Tautan yang beredar bukan milik instansi pemerintah manapun.

Modus penipuan serupa pernah dilaporkan sebelumnya. \
Korban diminta waspada dan tidak mengklik tautan mencurigakan.
""".strip()

pars = pisahkan_par(CONTOH_TEKS)
probs = infer_proba(pars)
print(f"Jumlah paragraf: {len(pars)}\n")

for metode in ["tfidf","lda","bertopic"]:
    print(f"{'─'*55}\n  {metode.upper()}\n{'─'*55}")
    hasil = ekstrak_topik(pars, metode=metode)
    for i, (par, prob, topik) in enumerate(zip(pars, probs, hasil)):
        verdict = "HOAKS" if prob["hoax"] >= 0.5 else "FAKTA"
        print(f"  P{i+1} [{verdict}|{prob['hoax']:.2f}] {topik['label']}")
        print(f"       kw: {topik['keywords'][:3]}")
    print()


---
## Evaluasi Topic Modelling V2

### Dua Jalur Wajib

#### Jalur 1 — Intrinsik: Topic Diversity
Mengukur keragaman keyword antar topik secara **independen dari label**.

#### Jalur 2 — Proxy: LogReg F1 + Confusion Matrix
> ⚠️ **PENTING — Baca sebelum menginterpretasi hasil:**
> Evaluasi ini **bukan** mengukur kualitas topik secara murni.
> Topic modelling bersifat **unsupervised** — tidak ada ground-truth untuk topik.
>
> Yang diukur: **seberapa informatif representasi topik terhadap label hoaks/non-hoaks**.
> Ini adalah evaluasi **proxy separabilitas**: apakah topik yang dihasilkan suatu metode
> dapat digunakan untuk memisahkan kedua kelas secara linear?
>
> F1 tinggi = representasi topik berkorelasi dengan label (informatif).
> F1 rendah = topik bersifat tematik netral (normal untuk topik murni).


In [ ]:
# ================================================================
# SEL 19 -- Evaluasi Intrinsik: Topic Diversity
# ================================================================
def hitung_diversity(metode, topk=10):
    """
    Topic Diversity = |kata unik| / (n_topik * topk)
    Mendekati 1.0 = topik sangat beragam (sedikit overlap keyword).
    """
    if metode == "lda" and model_lda is not None:
        feats = vec_lda.get_feature_names_out()
        all_words = []
        for topic in model_lda.components_:
            all_words.extend([feats[i] for i in topic.argsort()[-topk:][::-1]])
        n_t = len(model_lda.components_)
    elif metode == "bertopic" and model_bt is not None:
        all_words = []; n_t = 0
        info_loop = model_bt.get_topic_info()
        for t_id in info_loop["Topic"]:
            if t_id == -1: continue
            words = model_bt.get_topic(t_id)
            if words: all_words.extend([w for w,_ in words[:topk]]); n_t += 1
        if n_t == 0: return 0.0
    elif metode == "tfidf":
        sample = train_df_pra_oversampling["text"].sample(min(300, len(train_df_pra_oversampling)), random_state=42).tolist()
        hasil  = ekstrak_topik_tfidf(sample, topk=topk)
        all_words = [kw for h in hasil for kw in h["keywords"]]; n_t = len(hasil)
    else:
        return 0.0
    return round(min(1.0, len(set(all_words)) / max(1, n_t * topk)), 4) if all_words else 0.0

print("=== Jalur 1: Topic Diversity ===")
for m in ["tfidf","lda","bertopic"]:
    try:
        div = hitung_diversity(m, topk=10)
        print(f"  {m.upper():10s}: diversity = {div:.4f}")
    except Exception as e:
        print(f"  {m.upper():10s}: ERROR — {e}")
print("\nInterpretasi: mendekati 1.0 = topik beragam; mendekati 0 = banyak overlap keyword.")


In [ ]:
# ================================================================
# SEL 20 -- Evaluasi Proxy: LogReg F1 + Confusion Matrix
# ================================================================
# Lihat catatan interpretasi di sel markdown di atas!

def buat_fitur_proxy(metode, teks_list):
    """Buat representasi fitur topik untuk LogReg."""    if metode == "lda" and model_lda is not None:
        return model_lda.transform(vec_lda.transform(teks_list))
    elif metode == "bertopic" and model_bt is not None:
        global _emb_infer
        from sentence_transformers import SentenceTransformer
        if _emb_infer is None:
            dev = "cuda" if torch.cuda.is_available() else "cpu"
            _emb_infer = SentenceTransformer(MODEL_EMBEDDING_BT, device=dev)
        emb  = _emb_infer.encode(teks_list, batch_size=32, show_progress_bar=False,
                                 convert_to_numpy=True, normalize_embeddings=True)
        tids, _ = model_bt.transform(teks_list, embeddings=emb)
        n_t = max(2, max((t for t in tids if t != -1), default=1) + 1)
        X   = np.zeros((len(tids), n_t))
        for i, t_id in enumerate(tids):
            if 0 <= t_id < n_t: X[i, t_id] = 1.0
        return X
    elif metode == "tfidf":
        _vec_eval = TfidfVectorizer(max_features=500, stop_words=list(ID_STOPWORDS),
                                    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b")
        if not hasattr(buat_fitur_proxy, "_vec_fitted"):
            buat_fitur_proxy._vec_fitted = _vec_eval.fit(train_df_pra_oversampling["text"].astype(str).tolist())
        return buat_fitur_proxy._vec_fitted.transform(teks_list).toarray()
    return None

print("=== Jalur 2: Proxy Evaluasi (LogReg) ===")
print("(F1 = informatif terhadap label, BUKAN kualitas topik murni)\n")

test_s   = test_df.sample(min(400, len(test_df)), random_state=42)
X_tr_txt = train_df_pra_oversampling["text"].astype(str).tolist()
y_tr_lbl = train_df_pra_oversampling["label"].tolist()
X_te_txt = test_s["text"].astype(str).tolist()
y_te_lbl = test_s["label"].tolist()

hasil_proxy = []
for metode in ["tfidf","lda","bertopic"]:
    try:
        print(f"  Proses: {metode.upper()}...")
        Xtr = buat_fitur_proxy(metode, X_tr_txt)
        Xte = buat_fitur_proxy(metode, X_te_txt)
        if Xtr is None or Xte is None: print(f"  {metode.upper()}: skip\n"); continue
        clf = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
        clf.fit(Xtr, y_tr_lbl)
        y_pred = clf.predict(Xte)
        f1v = f1_score(y_te_lbl, y_pred, average="binary")
        cm  = confusion_matrix(y_te_lbl, y_pred)
        hasil_proxy.append({"metode": metode, "f1": f1v, "cm": cm})
        print(f"  {metode.upper()}: F1={f1v:.4f}\n  CM=\n{cm}\n")
    except Exception as e:
        print(f"  {metode.upper()}: ERROR — {e}\n")


In [ ]:
# ================================================================
# SEL 21 -- Plot Confusion Matrix Proxy
# ================================================================
if hasil_proxy:
    fig, axes = plt.subplots(1, len(hasil_proxy), figsize=(5*len(hasil_proxy), 4))
    if len(hasil_proxy) == 1: axes = [axes]
    for ax, hp in zip(axes, hasil_proxy):
        ConfusionMatrixDisplay(hp["cm"], display_labels=["not_hoax","hoax"]).plot(
            ax=ax, colorbar=False, cmap="Blues")
        ax.set_title(f"{hp['metode'].upper()}\nF1={hp['f1']:.3f}")
    plt.suptitle(
        "Proxy Evaluation — Topic Representation → LogReg\n"
        "Catatan: F1 mengukur informatif terhadap label, BUKAN kualitas topik murni",
        fontsize=9, y=1.05)
    plt.tight_layout()
    plt.savefig("/content/topic_eval_proxy.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Disimpan: /content/topic_eval_proxy.png")


In [ ]:
# ================================================================
# SEL 22 -- Tabel Perbandingan Lengkap
# ================================================================
div_scores = {}
for m in ["tfidf","lda","bertopic"]:
    try: div_scores[m] = hitung_diversity(m, topk=10)
    except: div_scores[m] = None

f1_map = {hp["metode"]: hp["f1"] for hp in hasil_proxy}

rows = []
for m in ["tfidf","lda","bertopic"]:
    rows.append({
        "Metode"       : m.upper(),
        "Diversity ↑"  : div_scores.get(m, "-"),
        "F1 Proxy ↑"   : round(f1_map[m],4) if m in f1_map else "-",
        "Artefak"      : {"tfidf":"Tidak perlu","lda":"lda_model.pkl",
                          "bertopic":"bertopic_model_v1/"}[m],
        "Kecepatan Inf.": {"tfidf":"⚡ Sangat cepat","lda":"⚡ Cepat",
                           "bertopic":"🐢 Lambat"}[m],
        "Cocok Deploy"  : {"tfidf":"✅","lda":"✅","bertopic":"⚠️ (butuh GPU)"}[m],
    })

df_cmp = pd.DataFrame(rows)
print("\n=== Tabel Perbandingan Topic Modelling V2 ===")
print(df_cmp.to_string(index=False))
print("\nCatatan:")
print("  Diversity: mendekati 1.0 = topik beragam.")
print("  F1 Proxy : mengukur separabilitas terhadap label (proxy), bukan kualitas topik murni.")
print("  Untuk deployment ringan: pilih TF-IDF atau LDA.")


---
## No-Regression Guarantee V2

1. **Arsitektur IndoBERT tidak diubah** — classifier biner dua kelas, split 70/15/15,
   oversampling hanya di train, semua identik dengan V1.
2. **Topic modelling adalah post-training** — tidak ada satu pun operasi topic
   modelling yang masuk ke dalam training loop atau mengubah gradient.
3. **Flags toggle aman** — `AKTIFKAN_LDA=False` / `AKTIFKAN_BERTOPIC=False` melewati
   semua kode terkait tanpa efek samping pada classifier.
4. **Fallback otomatis** — setiap metode topik memiliki fallback ke TF-IDF jika
   artefak tidak tersedia (aman untuk deployment minimal).
5. **Repo HF terpisah** — IndoBERT dan topic model artifacts di-push ke repo HF
   yang berbeda untuk memisahkan dependensi dan mempermudah versioning.
